In [2]:
from typing import Callable

import pandas as pd
import numpy as np

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchmetrics import Accuracy

In [30]:
class MNISTDatasetCsv(Dataset):
  def __init__(self, path: str):
    #df = pd.read_csv('../data/train_data.csv', header=None, dtype=float)
    #self.data = torch.tensor(df.values[:, 1:]).float().reshape(df.shape[0], 1, 28, 28)
    #self.labels = torch.tensor(df.iloc[:, 0]).long()
    data = np.load(path)
    self.data = torch.tensor(data[:, 1:]).float().reshape(data.shape[0], 1, 28, 28)
    self.labels = torch.tensor(data[:, 0]).long()

  def __len__(self) -> int:
    return self.labels.shape[0]

  def __getitem__(self, index) -> tuple[torch.Tensor, int]:
    return self.data[index], self.labels[index]


class MNISTClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
      nn.Conv2d(1, 8, kernel_size=3),
      nn.ReLU(),
      nn.Conv2d(8, 16, kernel_size=3),
      nn.ReLU(),
      nn.Flatten(),
      nn.Linear(9216, 10),  # 10 classes in total.
    )

  def forward(self, x):
    return self.model(x)
    

def train(dataloader, model, loss_fn, metrics_fn, optimizer, epoch, device):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        print(X.shape)

        pred = model(X)
        loss = loss_fn(pred, y)
        accuracy = metrics_fn(pred, y)

        # Backpropagation.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 10 == 0:
            loss, current = loss.item(), batch
            step = batch // 10 * (epoch + 1)
            #mlflow.log_metric("loss", f"{loss:2f}", step=step)
            #mlflow.log_metric("accuracy", f"{accuracy:2f}", step=step)
            print(f"{step} loss: {loss:2f} accuracy: {accuracy:2f} [{current} / {len(dataloader)-1}]")
    
    if batch > current:
      print(f"loss: {loss.item():2f} accuracy: {accuracy:2f} [{batch} / {len(dataloader)-1}]")

In [13]:
# Get cpu or gpu for training.
device = "cuda" if torch.cuda.is_available() else "cpu"

epochs = 1
loss_fn = nn.CrossEntropyLoss()
loss_fn = getattr(nn, 'CrossEntropyLoss')()
metric_fn = Accuracy(task="multiclass", num_classes=10).to(device)
model = MNISTClassifier().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

train_dataset = MNISTDatasetCsv('../data/train_data.npy')
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [15]:
for e in range(epochs): 
  train(train_dataloader, model, loss_fn, metric_fn, optimizer, e, device)

torch.Size([64, 1, 28, 28])
0 loss: 2.368251 accuracy: 0.062500 [0 / 78]
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
1 loss: 2.282413 accuracy: 0.125000 [10 / 78]
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
2 loss: 2.234808 accuracy: 0.218750 [20 / 78]
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 28])
torch.Size([64, 1, 28, 

In [3]:
model = MNISTClassifier()

input = np.random.uniform(size=[1, 28, 28])

print(input.shape)
model(input)

NameError: name 'MNISTClassifier' is not defined

In [9]:
input = np.random.uniform(size=[1, 28, 28])

In [5]:
input

array([[[0.17812655, 0.59621068, 0.28286878, 0.29957747, 0.74031333,
         0.18146342, 0.5674484 , 0.51948067, 0.90922918, 0.64063196,
         0.64617318, 0.09277892, 0.37434481, 0.5404316 , 0.8757257 ,
         0.97559803, 0.79250996, 0.2340716 , 0.18084264, 0.84285419,
         0.00457292, 0.33752995, 0.80283383, 0.07247548, 0.58186164,
         0.54949239, 0.41491617, 0.206112  ],
        [0.40633743, 0.48423267, 0.08766166, 0.66629148, 0.81170503,
         0.41835236, 0.02399027, 0.30773123, 0.14077183, 0.04171351,
         0.45880596, 0.04090673, 0.29949959, 0.74560164, 0.77382608,
         0.53649809, 0.19496699, 0.30964992, 0.74517637, 0.47606724,
         0.26161291, 0.63266369, 0.92625278, 0.94727812, 0.14078551,
         0.03777403, 0.37749901, 0.4153408 ],
        [0.59511712, 0.32261368, 0.60033138, 0.98746851, 0.43583088,
         0.59281921, 0.236465  , 0.19974748, 0.47462456, 0.27890029,
         0.22627749, 0.28298032, 0.32142213, 0.10864505, 0.00768069,
         0.

In [103]:
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

In [104]:
for x in training_data:
  print(x[0].shape)
  break

torch.Size([1, 28, 28])


In [111]:
for s in train_dataset:
  print(s[0].shape)
  break

torch.Size([1, 28, 28])


In [54]:
train_tensor = torch.tensor(train_data).float().reshape(train_data.shape[0], 28, 28)

In [55]:
train_data.shape

(5000, 28, 28)

In [70]:
train_df = pd.read_csv('../data/train_data.csv', header=None, dtype=float)
train_df.shape[0]

5000